In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/config.json
/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/tokenizer.json
/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/metadata.json
/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/model.weights.h5
/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/assets/tokenizer/merges.txt
/kaggle/input/models/keras/roberta/keras/roberta_base_en/3/assets/tokenizer/vocabulary.json
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 5

In [2]:
import transformers
import datasets
import torch
import os
import random
import numpy as np
import pandas as pd

import torch.nn.functional as F

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
#Config
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(DEVICE)

cuda


In [4]:
#loading dataset
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [5]:
def build_prompt(row):

    return f"""
Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

train_df["label"] = train_df["answer"].map(label2id)
train_df["text"] = train_df.apply(build_prompt, axis=1)

train_df[["text","label"]].head()

,text,label
0,\nQuestion:\nPick the best possible answer: Wh...,1
1,\nQuestion:\nWhat is accelerator-based light-i...,0
2,\nQuestion:\nDetermine the correct option: Wha...,2
3,\nQuestion:\nSelect the most accurate option: ...,1
4,\nQuestion:\nIdentify the correct statement: W...,0


In [6]:
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

train_dataset = Dataset.from_pandas(
    train_part[["text","label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_part[["text","label"]]
)

### Deberta-v3-base

In [7]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256

def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_tokenized = train_dataset.map(tokenize, batched=True)

valid_tokenized = valid_dataset.map(tokenize, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"])

valid_tokenized = valid_tokenized.remove_columns(["text"])

train_tokenized = train_tokenized.rename_column("label","labels")

valid_tokenized = valid_tokenized.rename_column("label","labels")


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=5
)
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc
    }

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias       

In [9]:
training_args = TrainingArguments(
    output_dir="./deberta_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    logging_steps=100,

    fp16=False,
    bf16=False,

    report_to="none"
)

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [10]:
import transformers
import torch
import accelerate

print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(torch.cuda.get_device_name(0))

5.0.0
2.10.0+cu128
1.13.0
Tesla T4


In [11]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=valid_tokenized,

    compute_metrics=compute_metrics
)

trainer.train()

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,nan,0.185000
2,1840.282031,nan,0.185000
3,1840.282031,nan,0.185000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

{'eval_loss': nan,
 'eval_accuracy': 0.185,
 'eval_runtime': 0.5739,
 'eval_samples_per_second': 348.467,
 'eval_steps_per_second': 12.196,
 'epoch': 3.0}

In [12]:
SAVE_DIR = "./deberta_finetuned"

trainer.save_model(SAVE_DIR)

tokenizer.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./deberta_finetuned/tokenizer_config.json',
 './deberta_finetuned/tokenizer.json')

In [13]:
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR
)

deberta_tokenizer = AutoTokenizer.from_pretrained(
    SAVE_DIR
)

deberta_model.to(DEVICE)

deberta_model.eval()

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [14]:
sample = valid_part.iloc[0]

text = sample["text"]

inputs = deberta_tokenizer(

    text,

    return_tensors="pt",

    truncation=True,

    max_length=256
)

inputs = {k:v.to(DEVICE) for k,v in inputs.items()}

with torch.no_grad():

    logits = deberta_model(**inputs).logits

prob = F.softmax(logits, dim=-1)

print(prob)

tensor([[nan, nan, nan, nan, nan]], device='cuda:0', dtype=torch.float16)


### RoBERTa

In [15]:
ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL
)

MAX_LEN = 256

def roberta_tokenize(example):

    return roberta_tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

roberta_train = train_dataset.map(
    roberta_tokenize,
    batched=True
)

roberta_valid = valid_dataset.map(
    roberta_tokenize,
    batched=True
)

roberta_train = roberta_train.remove_columns(
    ["text"]
)

roberta_valid = roberta_valid.remove_columns(
    ["text"]
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [16]:
roberta_train = roberta_train.rename_column(
    "label",
    "labels"
)

roberta_valid = roberta_valid.rename_column(
    "label",
    "labels"
)

roberta_train.set_format("torch")

roberta_valid.set_format("torch")

In [17]:
roberta_model = AutoModelForSequenceClassification.from_pretrained(

    ROBERTA_MODEL,

    num_labels=5
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
roberta_args = TrainingArguments(

    output_dir="./roberta_model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    logging_steps=100,

    fp16=False,

    bf16=False,

    report_to="none"
)

In [19]:
roberta_trainer = Trainer(

    model=roberta_model,

    args=roberta_args,

    train_dataset=roberta_train,

    eval_dataset=roberta_valid,

    compute_metrics=compute_metrics
)

roberta_trainer.train()
roberta_trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.657228,0.545000
2,2.590317,0.783485,0.920000
3,2.590317,0.256624,0.995000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.25662410259246826,
 'eval_accuracy': 0.995,
 'eval_runtime': 1.8058,
 'eval_samples_per_second': 110.752,
 'eval_steps_per_second': 3.876,
 'epoch': 3.0}

In [20]:
ROBERTA_SAVE = "./roberta_finetuned"

roberta_trainer.save_model(
    ROBERTA_SAVE
)

roberta_tokenizer.save_pretrained(
    ROBERTA_SAVE
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./roberta_finetuned/tokenizer_config.json',
 './roberta_finetuned/tokenizer.json')

In [21]:
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_SAVE
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_SAVE
)

roberta_model.to(DEVICE)

roberta_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [22]:
sample = valid_part.iloc[0]

text = sample["text"]

inputs = roberta_tokenizer(

    text,

    return_tensors="pt",

    truncation=True,

    max_length=256
)

inputs = {k:v.to(DEVICE) for k,v in inputs.items()}

with torch.no_grad():

    logits = roberta_model(**inputs).logits

prob = F.softmax(
    logits,
    dim=-1
)

print(prob)

tensor([[0.9224, 0.0111, 0.0253, 0.0166, 0.0246]], device='cuda:0')


In [23]:
label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

def predict_probs(model, tokenizer, text):

    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k:v.to(DEVICE) for k,v in inputs.items()}

    model.eval()

    with torch.no_grad():

        logits = model(**inputs).logits

    probs = F.softmax(
        logits,
        dim=-1
    )

    return probs.cpu().numpy()[0]

### Question - 1

In [24]:
row = train_df.iloc[25]

text = build_prompt(row)

deberta_probs = predict_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

idx = np.argmax(deberta_probs)

print("Answer :", label_map[idx])
print("Probability :", deberta_probs[idx])

Answer : A
Probability : nan


### Question - 2

In [25]:
roberta_probs = predict_probs(
    roberta_model,
    roberta_tokenizer,
    text
)

avg_probs = (
    deberta_probs +
    roberta_probs
)/2

idx = np.argmax(avg_probs)

print(label_map[idx])

A


### Question - 3

In [26]:
weighted_probs = (
    0.7*deberta_probs +
    0.3*roberta_probs
)

idx = np.argmax(weighted_probs)

print(label_map[idx])

A


### Question - 4

In [27]:
ranking = np.argsort(
    weighted_probs
)[::-1]

top3 = [
    label_map[i]
    for i in ranking[:3]
]

prediction = " ".join(top3)

print(prediction)

E D C


### Question - 5

In [28]:
submission_predictions = []

for _, row in test_df.iterrows():

    text = build_prompt(row)

    d = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    r = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    final = (
        0.7*d +
        0.3*r
    )

    rank = np.argsort(final)[::-1]

    top3 = " ".join(
        label_map[i]
        for i in rank[:3]
    )

    submission_predictions.append(top3)

submission = pd.DataFrame({

    "id":test_df["id"],

    "prediction":submission_predictions

})

submission.to_csv(
    "submission.csv",
    index=False
)

print("Rows :", len(submission))

Rows : 500


### Question - 6

In [29]:
changed = 0

for i in range(50):

    row = test_df.iloc[i]

    original = build_prompt(row)

    augmented = (
        "Answer the following multiple-choice question carefully:\n\n"
        + original
    )

    p1 = predict_probs(
        deberta_model,
        deberta_tokenizer,
        original
    )

    p2 = predict_probs(
        deberta_model,
        deberta_tokenizer,
        augmented
    )

    final = (
        p1+p2
    )/2

    if np.argmax(final)!=np.argmax(p1):
        changed+=1

print(changed)

0


### Question - 7

In [30]:
different = 0

for i in range(100):

    row = test_df.iloc[i]

    text = build_prompt(row)

    d = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    r = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble = (
        0.7*d+
        0.3*r
    )

    if np.argmax(d)!=np.argmax(ensemble):
        different+=1

print(different)

0


### Question - 8

In [31]:
gain = 0

for i in range(100):

    row = test_df.iloc[i]

    text = build_prompt(row)

    d = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    r = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble = (
        0.7*d+
        0.3*r
    )

    if ensemble.max()>d.max():
        gain+=1

print(gain)

0


### Question - 9 

In [32]:
changes = 0

for i in range(100):

    row = test_df.iloc[i]

    text = build_prompt(row)

    d = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    r = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble = (
        0.7*d+
        0.3*r
    )

    top3_d = np.argsort(d)[::-1][:3]

    top3_e = np.argsort(ensemble)[::-1][:3]

    if not np.array_equal(
        top3_d,
        top3_e
    ):
        changes+=1

print(changes)

0


### Question 10

In [33]:
def apk(actual,predicted,k=3):

    for i,p in enumerate(predicted[:k]):

        if p==actual:
            return 1/(i+1)

    return 0


scores=[]

validation = train_df.iloc[:100]

for _,row in validation.iterrows():

    text = build_prompt(row)

    d = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    r = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    ensemble = (
        0.7*d+
        0.3*r
    )

    ranking = np.argsort(
        ensemble
    )[::-1][:3]

    prediction = [
        label_map[i]
        for i in ranking
    ]

    actual = row["answer"]

    scores.append(
        apk(actual,prediction)
    )

print("MAP@3 =",round(np.mean(scores),4))

MAP@3 = 0.3067
